In [0]:
import requests
import json
from datetime import datetime, date
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, explode, from_json
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, ArrayType, MapType, TimestampType, BooleanType
)
 
# In Databricks the SparkSession is pre-created as `spark`.
# Uncomment the line below only when running locally:
# spark = SparkSession.builder.appName("OpenAPI_Demo").getOrCreate()
 
print("✅ Spark version:", spark.version)
print("✅ Notebook started at:", datetime.utcnow().isoformat(), "UTC")
 

In [0]:
# ─── Generic API helper ─────────────────────────────────────
def fetch_json(url: str, params: dict = None, timeout: int = 15) -> dict | list:
    """Fetch JSON from a URL with basic error handling."""
    resp = requests.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp.json()
 
 
# COMMAND ----------
# ══════════════════════════════════════════════════════════════
# 1. REAL-TIME WEATHER  –  Open-Meteo (no key needed)
#    Docs: https://open-meteo.com/en/docs
# ══════════════════════════════════════════════════════════════
 
CITIES = [
    {"name": "Sydney",        "lat": -33.87, "lon": 151.21},
    {"name": "New York",      "lat":  40.71, "lon": -74.01},
    {"name": "London",        "lat":  51.51, "lon":  -0.13},
    {"name": "Tokyo",         "lat":  35.68, "lon": 139.69},
    {"name": "Johannesburg",  "lat": -26.20, "lon":  28.04},
    {"name": "Kathmandu",  "lat": 27.71, "lon": 85.32},
]
 
weather_records = []
 
for city in CITIES:
    data = fetch_json(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude":              city["lat"],
            "longitude":             city["lon"],
            "current_weather":       "true",
            "hourly":                "temperature_2m,relativehumidity_2m,windspeed_10m",
            "forecast_days":         1,
            "timezone":              "auto",
        }
    )
    cw = data["current_weather"]
    weather_records.append({
        "city":              city["name"],
        "latitude":          float(city["lat"]),
        "longitude":         float(city["lon"]),
        "temperature_c":     float(cw["temperature"]),
        "windspeed_kmh":     float(cw["windspeed"]),
        "weathercode":       int(cw["weathercode"]),
        "is_day":            bool(cw["is_day"]),
        "fetched_at_utc":    datetime.utcnow().isoformat(),
    })
 
weather_schema = StructType([
    StructField("city",           StringType(),    False),
    StructField("latitude",       DoubleType(),    False),
    StructField("longitude",      DoubleType(),    False),
    StructField("temperature_c",  DoubleType(),    True),
    StructField("windspeed_kmh",  DoubleType(),    True),
    StructField("weathercode",    IntegerType(),   True),
    StructField("is_day",         BooleanType(),   True),
    StructField("fetched_at_utc", StringType(),    True),
])
 
df_weather = spark.createDataFrame(weather_records, schema=weather_schema)
df_weather.printSchema()
df_weather.show(truncate=False)

In [0]:
df_weather.describe()

In [0]:
df_weather.display()